# Step 1: Collect & Prepare (Thu Thập & Chuẩn Bị Dữ Liệu)
### Theo Quy Trình 8 Bước Từ Slide Bài Giảng Chapter 4 (NEU FIT)

---

### 1. Phân Tích Chuyên Sâu Cách Làm:
Trước khi viết bất kỳ dòng mã nào, Chapter 4 đặt ra **3 câu hỏi vàng**:
1. **Dự đoán cái gì? (What do we predict?)**: Dự báo doanh thu hàng ngày (`revenue`, đơn vị: triệu VNĐ) cho từng quán cà phê.
2. **Từ những cột nào? (From which columns?)**: Từ ngày (`date`), mã quán (`store_id`), khuyến mãi (`is_promo`), cùng các bảng liên kết phụ trợ: thông tin quán (`stores.csv`), thời tiết (`weather.csv`) và ngày lễ Tết (`holidays.csv`).
3. **Đánh giá bằng thước đo gì? (Measured by what?)**: Thước đo chính thức là **RMSLE** (Root Mean Squared Logarithmic Error) tính trên $\log(1 + \text{revenue})$:
   $$\text{RMSLE} = \sqrt{\frac{1}{N} \sum_{i=1}^N \left(\log(1 + \hat{y}_i) - \log(1 + y_i)\right)^2}$$

### 2. Dùng Những Gì Để Làm?
- **Thư viện chính:** `pandas` (hàm `pd.read_csv`, `pd.merge`).
- **Kỹ thuật dữ liệu:** Thiết kế sơ đồ quan hệ (Relational Schema), phép nối `how='left'` để bảo toàn dòng bảng chính, và xây dựng **Data Dictionary** chuẩn hóa.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Thêm đường dẫn project root vào sys.path
for p in [os.path.abspath(".."), os.path.abspath("."), os.path.abspath("../..")]:
    if os.path.exists(os.path.join(p, "data")):
        DATA_DIR = os.path.join(p, "data")
    if p not in sys.path and os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, p)

print(f">>> THƯ MỤC DỮ LIỆU ĐANG SỬ DỤNG: {DATA_DIR}")

# 1. Đọc các bảng dữ liệu
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
stores_df = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
weather_df = pd.read_csv(os.path.join(DATA_DIR, "weather.csv"))
holidays_df = pd.read_csv(os.path.join(DATA_DIR, "holidays.csv"))

print("KÍCH THƯỚC CÁC BẢNG NGUỒN:")
print(f"- train.csv    : {train_df.shape[0]:,} dòng x {train_df.shape[1]} cột")
print(f"- test.csv     : {test_df.shape[0]:,} dòng x {test_df.shape[1]} cột")
print(f"- stores.csv   : {stores_df.shape[0]:,} dòng x {stores_df.shape[1]} cột")
print(f"- weather.csv  : {weather_df.shape[0]:,} dòng x {weather_df.shape[1]} cột")
print(f"- holidays.csv : {holidays_df.shape[0]:,} dòng x {holidays_df.shape[1]} cột")

>>> THƯ MỤC DỮ LIỆU ĐANG SỬ DỤNG: E:\study\AI\test 1\Kaggle_Competition\data\cafe\data
KÍCH THƯỚC CÁC BẢNG NGUỒN:
- train.csv    : 26,632 dòng x 4 cột
- test.csv     : 3,869 dòng x 4 cột
- stores.csv   : 45 dòng x 8 cột
- weather.csv  : 1,593 dòng x 4 cột
- holidays.csv : 55 dòng x 3 cột


### 3. Thực Hiện Nối Bảng (Merge Tables) & Kiểm Tra Hình Dạng
Nguyên tắc từ Slide 13: *Compare shape before and after a merge: more rows means the key is repeated in the right table!*

In [2]:
# Đổi date sang chuỗi định dạng đồng nhất
train_df["date"] = pd.to_datetime(train_df["date"])
weather_df["date"] = pd.to_datetime(weather_df["date"])
holidays_df["date"] = pd.to_datetime(holidays_df["date"])

# 1. Nối train với stores qua key 'store_id'
m1 = train_df.merge(stores_df, on="store_id", how="left")
print(f"Shape trước khi merge stores: {train_df.shape} -> Sau khi merge: {m1.shape}")

# 2. Nối tiếp với weather qua cặp key ['date', 'city']
m2 = m1.merge(weather_df, on=["date", "city"], how="left")
print(f"Shape sau khi merge weather : {m2.shape}")

# 3. Nối tiếp với holidays qua key 'date'
# holidays_df có thể có nhiều sự kiện trong cùng 1 ngày, dedup để tránh nhân bản dòng
holidays_dedup = holidays_df.drop_duplicates(subset=["date"], keep="first")
full_merged = m2.merge(holidays_dedup, on="date", how="left")
print(f"Shape sau cùng khi merge holidays: {full_merged.shape}")

display(full_merged.head(3))

Shape trước khi merge stores: (26632, 4) -> Sau khi merge: (26632, 11)
Shape sau khi merge weather : (26632, 13)
Shape sau cùng khi merge holidays: (26632, 15)


,date,store_id,is_promo,revenue,city,district,store_type,area_m2,seats,has_parking,open_date,temp_c,rain_mm,holiday_name,type
0,2024-01-01,S02,0,6.44,Ha Noi,Thanh Xuan,residential,95.0,28,1,27/07/2022,15.9,0.0,New Year's Day,national_holiday
1,2024-01-01,S03,0,8.86,Ha Noi,Hai Ba Trung,office,120.0,38,0,12/03/2018,15.9,0.0,New Year's Day,national_holiday
2,2024-01-01,S04,0,17.08,Ha Noi,Tay Ho,street,101.0,41,0,17/04/2019,15.9,0.0,New Year's Day,national_holiday


### 4. Bảng Từ Điển Dữ Liệu Toàn Diện (Data Dictionary)
Theo chuẩn Chapter 4, việc lập Data Dictionary giúp toàn bộ team hiểu rõ ý nghĩa, đơn vị và kiểu dữ liệu trước khi đi vào phân tích.

In [3]:
data_dict = pd.DataFrame([
    {"Cột": "date", "Bảng Gốc": "train/test/weather", "Ý Nghĩa Nghiệp Vụ": "Ngày ghi nhận giao dịch", "Đơn Vị": "YYYY-MM-DD", "Kiểu": "datetime"},
    {"Cột": "store_id", "Bảng Gốc": "train/test/stores", "Ý Nghĩa Nghiệp Vụ": "Mã định danh cửa hàng", "Đơn Vị": "S01 - S45", "Kiểu": "category"},
    {"Cột": "is_promo", "Bảng Gốc": "train/test", "Ý Nghĩa Nghiệp Vụ": "Có chạy chương trình khuyến mãi hay không", "Đơn Vị": "0 (Không), 1 (Có)", "Kiểu": "int / bool"},
    {"Cột": "revenue", "Bảng Gốc": "train", "Ý Nghĩa Nghiệp Vụ": "Biến mục tiêu: Doanh thu trong ngày của quán", "Đơn Vị": "Triệu VNĐ", "Kiểu": "float (Target)"},
    {"Cột": "city", "Bảng Gốc": "stores/weather", "Ý Nghĩa Nghiệp Vụ": "Thành phố quán đặt cơ sở", "Đơn Vị": "Ha Noi, Ho Chi Minh, Da Nang", "Kiểu": "category"},
    {"Cột": "district", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Quận/huyện của cửa hàng", "Đơn Vị": "Tên quận", "Kiểu": "text"},
    {"Cột": "store_type", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Mô hình kinh doanh mặt bằng", "Đơn Vị": "mall, office, street, residential, university", "Kiểu": "category"},
    {"Cột": "area_m2", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Diện tích mặt sàn kinh doanh", "Đơn Vị": "m2", "Kiểu": "float"},
    {"Cột": "seats", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Số lượng ghế ngồi phục vụ", "Đơn Vị": "Số chỗ", "Kiểu": "int"},
    {"Cột": "has_parking", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Có bãi giữ xe máy/ô tô không", "Đơn Vị": "0 (Không), 1 (Có)", "Kiểu": "int / bool"},
    {"Cột": "open_date", "Bảng Gốc": "stores", "Ý Nghĩa Nghiệp Vụ": "Ngày quán bắt đầu khai trương", "Đơn Vị": "YYYY-MM-DD", "Kiểu": "datetime"},
    {"Cột": "temp_c", "Bảng Gốc": "weather", "Ý Nghĩa Nghiệp Vụ": "Nhiệt độ trung bình trong ngày", "Đơn Vị": "Độ Celsius (°C)", "Kiểu": "float"},
    {"Cột": "rain_mm", "Bảng Gốc": "weather", "Ý Nghĩa Nghiệp Vụ": "Lượng mưa đo đạc trong ngày", "Đơn Vị": "mm", "Kiểu": "float"},
    {"Cột": "holiday_name", "Bảng Gốc": "holidays", "Ý Nghĩa Nghiệp Vụ": "Tên ngày lễ hoặc sự kiện", "Đơn Vị": "Chuỗi văn bản", "Kiểu": "text"},
    {"Cột": "type", "Bảng Gốc": "holidays", "Ý Nghĩa Nghiệp Vụ": "Phân loại kỳ nghỉ", "Đơn Vị": "national_holiday, event", "Kiểu": "category"}
])

display(data_dict)
print(">>> HOÀN THÀNH STEP 1: ĐÃ XÁC ĐỊNH BÀI TOÁN, KẾT NỐI DỮ LIỆU VÀ LẬP TỪ ĐIỂN DỮ LIỆU!")

,Cột,Bảng Gốc,Ý Nghĩa Nghiệp Vụ,Đơn Vị,Kiểu
0,date,train/test/weather,Ngày ghi nhận giao dịch,YYYY-MM-DD,datetime
1,store_id,train/test/stores,Mã định danh cửa hàng,S01 - S45,category
2,is_promo,train/test,Có chạy chương trình khuyến mãi hay không,"0 (Không), 1 (Có)",int / bool
3,revenue,train,Biến mục tiêu: Doanh thu trong ngày của quán,Triệu VNĐ,float (Target)
4,city,stores/weather,Thành phố quán đặt cơ sở,"Ha Noi, Ho Chi Minh, Da Nang",category
5,district,stores,Quận/huyện của cửa hàng,Tên quận,text
6,store_type,stores,Mô hình kinh doanh mặt bằng,"mall, office, street, residential, university",category
7,area_m2,stores,Diện tích mặt sàn kinh doanh,m2,float
8,seats,stores,Số lượng ghế ngồi phục vụ,Số chỗ,int
9,has_parking,stores,Có bãi giữ xe máy/ô tô không,"0 (Không), 1 (Có)",int / bool


>>> HOÀN THÀNH STEP 1: ĐÃ XÁC ĐỊNH BÀI TOÁN, KẾT NỐI DỮ LIỆU VÀ LẬP TỪ ĐIỂN DỮ LIỆU!
